# 제주도 퇴근시간 버스 승차인원 예측

## 프로젝트 개요
| 항목 | 내용 |
|------|------|
| 대회 | Dacon 제주 버스탑승객수예측 AI 경진대회 |
| 목표 | 18~20시 저녁 승차인원(`18~20_ride`) 예측 |
| 평가지표 | RMSE (Root Mean Squared Error) |
| 학습 데이터 | 2019년 9월 (30일, 415,423행) |
| 예측 대상 | 2019년 10월 1~16일 (228,170행) |
| 팀원 | 5명 |

## 데이터셋 전체 목록

### 사용한 데이터
| 파일 | 설명 | 크기 | 사용 이유 |
|------|------|------|----------|
| `train.csv` | 9월 버스 승하차 데이터 (학습용) | 415,423행 × 21열 | 핵심 학습 데이터. 시간대별 승차/하차 + 타겟(18~20_ride) 포함 |
| `test.csv` | 10월 버스 승하차 데이터 (예측 대상) | 228,170행 × 20열 | 제출용 예측 대상. 타겟 컬럼 없음 |
| `bus_bts.csv` | 버스카드(BTS) 탑승 기록 | 2,409,414행 × 13열 | 카드 기반 승하차 기록. 아침 하차(bts_morning_getoff), 배차간격(bus_interval), 통근자/관광객 비율 등 핵심 피처 추출에 사용 |
| `OBS_ASOS_TIM_*.csv` (9월) | 기상청 ASOS 시간별 관측 (9월) | 시간별 | 기온, 강수량, 풍속, 일조시간 → 일별/저녁시간 집계하여 기상 피처 7개 생성 |
| `OBS_ASOS_TIM_*_10월.csv` (10월) | 기상청 ASOS 시간별 관측 (10월) | 시간별 | test 기간(10월)의 기상 데이터. train과 동일 방식으로 집계 |
| `submission_sample.csv` | 제출 양식 | 228,170행 × 2열 | id + 18~20_ride 예측값 기입용 |

### 사용하지 않은 데이터
| 파일 | 설명 | 미사용 이유 |
|------|------|------------|
| `jeju_financial_life_data.csv` | 제주 동별 금융생활 데이터 (소득, 직업, 지출 등) | 정류장 좌표 → 동 매칭을 위한 주소 데이터(`data_address.csv`)가 없어서 정확한 매칭이 어려움. 좌표 기반 근사 매칭을 별도 실험(plan24)했으나 최종 모델에는 미포함 |
| `행정_법정동 중심좌표.xlsx` | 행정동/법정동 중심 좌표 | 동 단위 외부 데이터 연결용이지만, 현재 모델에서는 정류장 좌표를 직접 사용하는 방식(dist_to_dong)으로 대체하여 미사용 |
---

## 전체 파이프라인

```
1. 라이브러리 & 데이터 로드
   |
2. EDA (탐색적 데이터 분석)
   ├── Chapter 1. 언제 타는가? (요일/공휴일/시간대 패턴)
   ├── Chapter 2. 오전과 저녁의 관계 (아침 승차 vs 저녁 승차)
   ├── Chapter 3. 어디서 타는가? (정류장/노선별 패턴)
   ├── Chapter 4. 누가 타는가? (관광객/직장인/학생/현지인)
   └── Chapter 5. 날씨의 영향 (강수/기온/태풍)
   |
3. 전처리 (기상 데이터, 날짜 피처, BTS merge)
   |
4. 특성 엔지니어링 (팀원 5명 피처 통합 → 59개)
   ├── 승차/하차/순유입 피처 (팀원 1,2,3,4,5 공통)
   ├── Target Encoding (팀원 1,3,5 공통)
   ├── BTS 카드 피처 (팀원 2 핵심)
   ├── 정류장 통계 + 상호작용 (팀원 4,5)
   └── 팀원 고유 피처 4개 (관광객/현지인/거리/경과일수)
   |
5. 모델링 (LightGBM + RandomForest + XGBoost)
   ├── Optuna 하이퍼파라미터 튜닝 (시간 기반 CV)
   ├── 3모델 × 3시드 앙상블
   └── 55개 vs 59개 비교 + 블렌딩
   |
6. 제출 파일 생성
```


| 컬럼명 | 설명 |
|------|------|
| id | 데이터 고유 식별자 |
| date | 버스 이용 날짜 |
| bus_route_id | 버스 노선 ID |
| in_out | 시내 / 시외 구분 |
| station_code | 정류장 코드 |
| station_name | 정류장 이름 |
| latitude | 정류장 위도 |
| longitude | 정류장 경도 |
| 6~7_ride | 6~7시 승차 인원 |
| 7~8_ride | 7~8시 승차 인원 |
| 8~9_ride | 8~9시 승차 인원 |
| 9~10_ride | 9~10시 승차 인원 |
| 10~11_ride | 10~11시 승차 인원 |
| 11~12_ride | 11~12시 승차 인원 |
| 6~7_takeoff | 6~7시 하차 인원 |
| 7~8_takeoff | 7~8시 하차 인원 |
| 8~9_takeoff | 8~9시 하차 인원 |
| 9~10_takeoff | 9~10시 하차 인원 |
| 10~11_takeoff | 10~11시 하차 인원 |
| 11~12_takeoff | 11~12시 하차 인원 |
| 18~20_ride | 18~20시 승차 인원 (예측 대상) |

------------------------------------------------------------------------------------
대회 평가지표는 RMSE이지만, 학습 손실함수로는 MAE(L1)를 선택했습니다.

본 데이터는 타겟의 약 71%가 0에 집중된 zero-inflated 분포와
일부 큰 값을 포함하는 long-tail 구조를 가지고 있습니다.

RMSE는 오차를 제곱하는 특성상 큰 값에서 발생하는 오차에 대해
gradient가 크게 증가하며, 이로 인해 모델이 소수의 큰 값 샘플에

과도하게 최적화되는 경향이 나타났습니다.

이 과정에서 전체 분포에 대한 균형 잡힌 학습이 어려워지고
일반화 성능이 저하되는 문제가 발생했습니다.

이를 완화하기 위해 이상치에 덜 민감한 MAE를 손실함수로 사용하고,
동시에 log1p 변환을 통해 타겟 분포의 비대칭성과 스케일을 압축했습니다.

이 조합은 전체 데이터 분포를 보다 안정적으로 학습하게 했으며,
결과적으로 RMSE 기준에서도 더 낮은 오차를 기록했습니다.

"MAE가 이론적으로 유리한 구조라 MAE를 우선 적용했고, 결과가 좋아서 그대로 진행했습니다" 

=======================================================================================================

=======================================================================================================

---
# 1. 라이브러리 & 데이터 로드


In [ ]:
# ============================================================
# [셀 1] 라이브러리 불러오기 & 한글 폰트 설정
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import seaborn as sns
import warnings
import os
import matplotlib as mpl


# 한글 폰트 설정 (나눔고딕)
plt.rc('font', family='NanumGothic')

# 마이너스(-) 기호 폰트 깨짐 방지
mpl.rcParams['axes.unicode_minus'] = False

# (선택) 레티나 디스플레이 설정으로 글씨를 더 선명하게 표시
from IPython.display import set_matplotlib_formats
set_matplotlib_formats('retina')

#==================================#
# 폰트 적용 확인용 간단한 테스트 그래프  #
#==================================#
plt.figure(figsize=(5, 3))
plt.title('한글 폰트 인식 테스트용 그래프')
plt.plot([-3, 0, 3], [-3, 0, 3], label='테스트 선')
plt.xlabel('X 축 데이터')
plt.ylabel('Y 축 데이터')
plt.legend()
plt.show()



In [ ]:
# 폰트 설정은 위 셀에서 완료
print(f'현재 폰트: {plt.rcParams["font.family"]}')


In [ ]:
# ============================================================
# 셀 1. 라이브러리 임포트
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import gc

from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error

import lightgbm as lgb
import optuna

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)

SEED = 42
np.random.seed(SEED)
print('Libraries loaded.')

In [ ]:
# ============================================================
# 셀 2. 데이터 로드
# ============================================================

DATA_DIR = 'data/bus/'

train = pd.read_csv(f'{DATA_DIR}train.csv')
test  = pd.read_csv(f'{DATA_DIR}test.csv')
bus_bts = pd.read_csv(f'{DATA_DIR}bus_bts.csv')
submission = pd.read_csv(f'{DATA_DIR}submission_sample.csv')

weather_sep = pd.read_csv(f'{DATA_DIR}OBS_ASOS_TIM_20260309132252.csv', encoding='euc-kr')
weather_oct = pd.read_csv(f'{DATA_DIR}OBS_ASOS_TIM_20260309155017_10월.csv', encoding='euc-kr')

TARGET = '18~20_ride'

print(f'Train: {train.shape}, Test: {test.shape}, BTS: {bus_bts.shape}')

In [ ]:
# ============================================================
# [셀 2-1] 날씨 데이터 일별 집계 (EDA 시각화용)
# - cell[7]에서 로드한 weather_sep, weather_oct 활용
# ============================================================

weather_raw = pd.concat([weather_sep, weather_oct], ignore_index=True)
weather_raw['date'] = pd.to_datetime(weather_raw['일시']).dt.strftime('%Y-%m-%d')

weather_daily = weather_raw.groupby('date').agg(
    강수량_합계=('강수량(mm)', lambda x: pd.to_numeric(x, errors='coerce').fillna(0).sum()),
    기온_평균=('기온(°C)', lambda x: pd.to_numeric(x, errors='coerce').fillna(0).mean()),
    풍속_최대=('풍속(m/s)', lambda x: pd.to_numeric(x, errors='coerce').fillna(0).max()),
    풍속_평균=('풍속(m/s)', lambda x: pd.to_numeric(x, errors='coerce').fillna(0).mean()),
    일조_합계=('일조(hr)', lambda x: pd.to_numeric(x, errors='coerce').fillna(0).sum()),
).reset_index()

typhoon_days = ['2019-09-22', '2019-09-23']
weather_daily['태풍여부'] = weather_daily['date'].isin(typhoon_days)
weather_daily['강수여부'] = weather_daily['강수량_합계'] > 1.0

print(f'날씨 데이터: {weather_daily["date"].min()} ~ {weather_daily["date"].max()} ({len(weather_daily)}일)')
display(weather_daily.head(5))


In [ ]:
# ============================================================
# [셀 2-2] 데이터 기본 정보 & 결측치 확인
# - 각 컬럼의 타입, 결측치 개수 파악
# ============================================================

print('=== 기본 정보 ===')
print(train.info())
print()

# 결측치 개수 및 비율 계산
missing = pd.DataFrame({
    '결측치 수': train.isnull().sum(),
    '결측치 비율(%)': (train.isnull().sum() / len(train) * 100).round(2)
})
missing = missing[missing['결측치 수'] > 0].sort_values('결측치 비율(%)', ascending=False)

print('=== 결측치 현황 ===')
if len(missing) == 0:
    print('결측치 없음 ✅')
else:
    print(missing)

print()
print('=== 기술통계 ===')
display(train.describe().round(2))

In [ ]:
# ============================================================
# 셀 3. 타겟 분포 확인 + EDA용 변수 정의
# ============================================================

print('=== 타겟 분포 확인 ===')
print(train[TARGET].describe())
print(f'0인 비율: {(train[TARGET] == 0).mean():.4f}')
print(f'왜도: {train[TARGET].skew():.2f}')

# EDA 셀에서 사용할 상수 정의
TARGET_COL         = TARGET
DATE_COL           = 'date'

# EDA용 합산 컬럼
_ride_cols    = ['6~7_ride', '7~8_ride', '8~9_ride', '9~10_ride', '10~11_ride', '11~12_ride']
_takeoff_cols = ['6~7_takeoff', '7~8_takeoff', '8~9_takeoff', '9~10_takeoff', '10~11_takeoff', '11~12_takeoff']

if 'morning_ride_total' not in train.columns:
    train['morning_ride_total'] = train[_ride_cols].sum(axis=1)
if 'morning_takeoff_total' not in train.columns:
    train['morning_takeoff_total'] = train[_takeoff_cols].sum(axis=1)

MORNING_RIDE_COL   = 'morning_ride_total'
MORNING_ALIGHT_COL = 'morning_takeoff_total'

print('EDA 변수 정의 완료')


=======================================================================================================

=======================================================================================================

---
# 2. EDA (탐색적 데이터 분석)

팀원 5명이 각자 진행한 EDA를 통합한 결과입니다.
EDA를 통해 발견한 핵심 인사이트가 이후 특성 엔지니어링의 근거가 되었습니다.


### Chapter 1. 언제 타는가? : 시간과 이동 목적의 변화

In [ ]:
# ============================================================
# [셀 8] 요일별 승차 인원 패턴
# - teamplan1에서 date_dt, dayofweek는 이미 있음
# ============================================================

# teamplan1의 date_dt 재사용 (없으면 생성)
if 'date_dt' not in train.columns:
    train['date_dt'] = pd.to_datetime(train[DATE_COL])
train['date_parsed'] = train['date_dt']  # EDA 호환용 별칭

if '요일' not in train.columns:
    train['요일'] = train['date_dt'].dt.dayofweek
train['요일명'] = train['date_dt'].dt.day_name().map({
    'Monday': '월', 'Tuesday': '화', 'Wednesday': '수',
    'Thursday': '목', 'Friday': '금', 'Saturday': '토', 'Sunday': '일'
})
train['평일여부'] = train['요일'].apply(lambda x: '평일' if x < 5 else '주말')

요일_순서 = ['월', '화', '수', '목', '금', '토', '일']
요일_색상 = ['steelblue']*5 + ['tomato']*2

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('요일별 퇴근시간 승차 인원', fontsize=15, fontweight='bold')

weekday_mean = train.groupby('요일명')[TARGET_COL].mean().reindex(요일_순서)
bars = axes[0].bar(요일_순서, weekday_mean.values, color=요일_색상, edgecolor='white', width=0.7)
axes[0].set_xlabel('요일')
axes[0].set_ylabel('평균 퇴근 승차 인원')
axes[0].set_title('요일별 평균 승차 인원')
for bar, val in zip(bars, weekday_mean.values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
                 f'{val:.1f}', ha='center', va='bottom', fontsize=10)
axes[0].legend(handles=[
    plt.Rectangle((0,0),1,1, color='steelblue', label='평일'),
    plt.Rectangle((0,0),1,1, color='tomato',    label='주말')
])

data_weekday = [train[train['요일명'] == d][TARGET_COL].dropna().values for d in 요일_순서]
bp = axes[1].boxplot(data_weekday, labels=요일_순서, patch_artist=True,
                     medianprops=dict(color='black', linewidth=2))
for patch, color in zip(bp['boxes'], 요일_색상):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
axes[1].set_xlabel('요일')
axes[1].set_ylabel('퇴근 승차 인원')
axes[1].set_title('요일별 분포 (박스플롯)')

plt.tight_layout()
plt.show()

print('--- 요일별 평균 승차 인원 ---')
print(weekday_mean.round(2))

In [ ]:
# ============================================================
# [셀 11] 6위: 날짜별 / 공휴일 영향 분석
# - 9월 추석 연휴 / 10월 개천절·한글날 영향
# ============================================================

# 공휴일 목록 (9월 train 기준)
holidays_sep = ['2019-09-12', '2019-09-13', '2019-09-14']  # 추석 연휴
all_holidays  = holidays_sep

# 공휴일 여부 컬럼 추가
train['date_str'] = train['date_parsed'].dt.strftime('%Y-%m-%d')
train['공휴일']   = train['date_str'].isin(all_holidays)

# 날짜별 전체 합산 승차 인원
daily = train.groupby(['date_str', 'date_parsed', '평일여부', '공휴일'])[TARGET_COL].sum().reset_index()
daily = daily.sort_values('date_parsed').reset_index(drop=True)

fig, axes = plt.subplots(2, 1, figsize=(18, 12))
fig.suptitle('6위: 날짜별 / 공휴일 퇴근시간 총 승차 인원 추이', fontsize=15, fontweight='bold')

# ① 날짜별 총 승차 인원 추이
colors = ['tomato' if h else ('lightblue' if w == '주말' else 'steelblue')
          for h, w in zip(daily['공휴일'], daily['평일여부'])]
axes[0].bar(range(len(daily)), daily[TARGET_COL], color=colors, edgecolor='none', alpha=0.8)
axes[0].plot(range(len(daily)), daily[TARGET_COL], color='black', linewidth=0.8, alpha=0.5)

# 공휴일 레이블
for idx, row in daily[daily['공휴일']].iterrows():
    axes[0].annotate(
        row['date_str'][5:],
        xy=(idx, row[TARGET_COL]),
        xytext=(idx, row[TARGET_COL] + daily[TARGET_COL].max() * 0.05),
        fontsize=8, color='red', ha='center'
    )

axes[0].set_xticks(range(0, len(daily), 3))
axes[0].set_xticklabels(daily['date_str'].iloc[::3], rotation=45, fontsize=7)
axes[0].set_ylabel('총 승차 인원')
axes[0].set_title('날짜별 총 퇴근 승차 인원')
axes[0].legend(handles=[
    plt.Rectangle((0,0),1,1, color='steelblue', alpha=0.8, label='평일'),
    plt.Rectangle((0,0),1,1, color='lightblue', alpha=0.8, label='주말'),
    plt.Rectangle((0,0),1,1, color='tomato',    alpha=0.8, label='공휴일')
])

# ② 평일 / 주말 / 공휴일 평균 비교
train['구분'] = train.apply(
    lambda r: '공휴일' if r['공휴일'] else r['평일여부'], axis=1
)
cat_mean   = train.groupby('구분')[TARGET_COL].mean()
cat_colors = {'평일': 'steelblue', '주말': 'lightblue', '공휴일': 'tomato'}
bars = axes[1].bar(
    cat_mean.index,
    cat_mean.values,
    color=[cat_colors.get(k, 'gray') for k in cat_mean.index],
    edgecolor='white', width=0.5
)
for bar, val in zip(bars, cat_mean.values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{val:.2f}', ha='center', va='bottom', fontsize=12, fontweight='bold')
axes[1].set_ylabel('평균 퇴근 승차 인원 (정류소당)')
axes[1].set_title('평일 / 주말 / 공휴일 평균 승차 인원 비교')

plt.tight_layout()
plt.savefig('07_holiday_effect.png', dpi=150, bbox_inches='tight')
plt.show()
train.drop('구분', axis=1, inplace=True)

### Chapter 2. 오전과 저녁 승차량의 선형적 상관관계 

In [ ]:
# ============================================================
# [Chapter 2 - Step 2] 🥇 1위: 출근시간 승차 인원 vs Target
# - morning_ride(출근 총 승차) vs 18~20_ride
# - ① 산점도 + 추세선 / ② 하루 평균 오전 vs 퇴근 승차 인원 비율(파이 차트)
# ============================================================
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle('🥇 1위: 오전 시간대 활동량과 퇴근 시간 승차량 분석', fontsize=16, fontweight='bold')

# --- ① 전체 산점도 + 추세선 (기존 유지) ---
corr = train[[MORNING_RIDE_COL, TARGET_COL]].corr().iloc[0, 1]
axes[0].scatter(train[MORNING_RIDE_COL], train[TARGET_COL],
                alpha=0.3, color='steelblue', s=10)

# 추세선 (결측치 제거 후 계산)
mask = train[MORNING_RIDE_COL].notna() & train[TARGET_COL].notna()
z = np.polyfit(train.loc[mask, MORNING_RIDE_COL], train.loc[mask, TARGET_COL], 1)
p = np.poly1d(z)
x_line = np.linspace(train[MORNING_RIDE_COL].min(), train[MORNING_RIDE_COL].max(), 100)
axes[0].plot(x_line, p(x_line), 'r--', linewidth=3, label='추세선')

axes[0].set_xlabel('출근/오전 시간(6~12시) 누적 승차 인원', fontsize=12)
axes[0].set_ylabel('퇴근 시간(18~20시) 승차 인원', fontsize=12)
axes[0].set_title(f'오전 vs 저녁 승차량 산점도 (Pearson r = {corr:.3f})', fontsize=14)
axes[0].legend(fontsize=12)
axes[0].text(0.05, 0.9, f'r = {corr:.3f}', transform=axes[0].transAxes, fontsize=14, fontweight='bold',
             color='red', bbox=dict(boxstyle='round', facecolor='white', alpha=0.9, edgecolor='gray'))


# --- ② 하루 평균 승차 인원 기준 파이 차트 ---
# 날짜별로 먼저 총합을 구한 뒤, 평균(하루 평균)을 계산합니다.
daily_morning_avg = train.groupby(DATE_COL)[MORNING_RIDE_COL].sum().mean()
daily_evening_avg = train.groupby(DATE_COL)[TARGET_COL].sum().mean()

sizes = [daily_morning_avg, daily_evening_avg]
labels = ['하루 평균 오전(6~12시) 승차', '하루 평균 퇴근(18~20시) 승차']
colors = ['steelblue', 'tomato']
explode = (0.05, 0) # 오전 파이 조각을 살짝 떼어내어 강조

# 파이 차트 그리기
axes[1].pie(sizes, explode=explode, labels=labels, colors=colors, 
            autopct='%1.1f%%', shadow=True, startangle=90, 
            textprops={'fontsize': 13, 'fontweight': 'bold'})
axes[1].set_title('제주도 하루 평균 오전 vs 저녁 승차 인원 비율', fontsize=14)

# 파이 차트 아래에 실제 하루 평균 명수 텍스트 추가
axes[1].text(0, -1.3, f"(하루 평균 - 오전: {int(daily_morning_avg):,}명 / 퇴근: {int(daily_evening_avg):,}명)", 
             ha='center', fontsize=12, fontweight='bold', color='gray')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig('05_morning_vs_evening_daily_pie.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 분석할 시간대별 승차 인원 컬럼 목록
ride_cols = ['6~7_ride', '7~8_ride', '8~9_ride', '9~10_ride', '10~11_ride', '11~12_ride', '18~20_ride']

# Boxplot 시각화
plt.figure(figsize=(12, 6))
sns.boxplot(data=train[ride_cols])
plt.title('시간대별 승차 인원 분포 및 이상치 확인 (Train Data)', fontsize=15)
plt.ylabel('승차 인원 수 (명)')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# 타겟 변수(18~20_ride)에서 값이 가장 큰 상위 10개 데이터 확인
print("[ 18~20_ride 기준 극단치 상위 5개 데이터 ]")
display(train.nlargest(5, '18~20_ride')[['date', 'bus_route_id', 'station_name', '18~20_ride']])

### Chapter 3. 어디서 타고 내리는가? : 정류장의 입지와 역할 규명

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 노선별 평균 승차량 계산 및 상위 10개 추출
# top_routes = train.groupby('bus_route_id')['18~20_ride'].mean().sort_values(ascending=False).head(10)
top_routes = train.groupby('station_name')['18~20_ride'].mean().sort_values(ascending=False).head(10)

# 2. 시각화
plt.figure(figsize=(12, 6))
sns.barplot(x=top_routes.index.astype(str), y=top_routes.values, palette='viridis')
plt.xticks(rotation=45)
plt.title('퇴근 시간대 승차량 상위 10개 노선')
plt.ylabel('평균 승차 인원')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 정류장별 평균 승차량 계산
st_mean = train.groupby(['station_name', 'latitude', 'longitude'])['18~20_ride'].mean().reset_index()

# 2. 시각화 설정
plt.figure(figsize=(14, 10))
# 폰트는 셀 1에서 설정 완료

# 3. 보정된 산점도 그리기
# 'YlOrRd'(노랑-주황-빨강) 또는 'rocket' 팔레트를 쓰면 높은 수치가 아주 진하게 보입니다.
scatter = sns.scatterplot(data=st_mean, x='longitude', y='latitude', 
                          size='18~20_ride', hue='18~20_ride',
                          sizes=(40, 800), # 원의 최소/최대 크기를 키움
                          alpha=0.7,       # 투명도를 높여 색을 진하게 함
                          palette='YlOrRd', # 진한 빨간색 계열 팔레트
                          edgecolor='black', # 원 테두리를 검정으로 설정해 가시성 확보
                          linewidth=0.5)

plt.title('제주도 정류장별 퇴근 시간대 평균 승차량 (색상 보정)', fontsize=16, pad=20)
plt.xlabel('경도', fontsize=12)
plt.ylabel('위도', fontsize=12)

# 범례 위치 조정
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', title='평균 승차량')
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

### Chapter 4. 이용객 유형에 따른 노선별 선호도 분석

In [ ]:
import pandas as pd

# ----------------------------------------------------
# 1. 기본 설정 및 날짜/시간 추출
# ----------------------------------------------------
bus_bts['date'] = pd.to_datetime(bus_bts['geton_date'])
bus_bts['hour'] = pd.to_datetime(bus_bts['geton_time']).dt.hour

# ----------------------------------------------------
# 2. 관광객 판별 (이용 기간 5일 이하)
# ----------------------------------------------------
date_range = bus_bts.groupby('user_card_id')['date'].agg(['min','max'])
date_range['days_diff'] = (date_range['max'] - date_range['min']).dt.days
tourist_cards = date_range[date_range['days_diff'] <= 5].index

# ----------------------------------------------------
# 3. 직장인 판별 (아침 6~10시 동일 노선 반복 탑승)
# ----------------------------------------------------
morning_bts = bus_bts[(bus_bts['hour'] >= 6) & (bus_bts['hour'] <= 10)]
commute = morning_bts.groupby(['user_card_id','bus_route_id','hour']).size().reset_index(name='count')
worker_cards = commute[commute['count'] >= 5]['user_card_id'].unique()

# ----------------------------------------------------
# 4. 🎓 대학생 판별 (아침 7~11시에 '대학교' 정류장에서 하차)
# ----------------------------------------------------
# 하차 정류장 이름에 '대학교', '대학', '제주대', '한라대', '국제대' 등이 포함되지만,
# '병원'이라는 단어가 들어간 정류장(예: 제주대학교병원)은 제외하고 코드 추출
univ_stations = bus_bts[
    (bus_bts['getoff_station_name'].str.contains('대학교|대학|제주대|한라대|국제대', na=False)) & 
    (~bus_bts['getoff_station_name'].str.contains('병원', na=False))
]['getoff_station_code'].unique()

# 7시 ~ 11시 사이에 대학교 정류장에서 하차한 기록만 필터링 (등교 패턴)
univ_morning_bts = bus_bts[(bus_bts['hour'] >= 7) & (bus_bts['hour'] <= 11)]
univ_rides = univ_morning_bts[univ_morning_bts['getoff_station_code'].isin(univ_stations)]

# 해당 승객(카드)이 대학교 정류장에서 몇 번 내렸는지 집계
univ_commute = univ_rides.groupby('user_card_id').size().reset_index(name='count')

# 등교 목적으로 5번 이상 하차했다면 '대학생'으로 간주!
student_cards = univ_commute[univ_commute['count'] >= 5]['user_card_id'].unique()

# ----------------------------------------------------
# 5. 최종 user_type 부여 (현지인 -> 관광객 -> 직장인 -> 대학생 덮어쓰기)
# ----------------------------------------------------
bus_bts['user_type'] = '현지인'
bus_bts.loc[bus_bts['user_card_id'].isin(tourist_cards), 'user_type'] = '관광객'
bus_bts.loc[bus_bts['user_card_id'].isin(worker_cards), 'user_type'] = '직장인'

# 대학생이 가장 나중에 덮어씌워지도록 배치하여, 직장인 조건과 겹칠 경우 대학생으로 우선 분류!
bus_bts.loc[bus_bts['user_card_id'].isin(student_cards), 'user_type'] = '대학생'

print("✨ 관광객/직장인/대학생/현지인 4단계 분류 완료!")
print(bus_bts['user_type'].value_counts())

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정
# 폰트는 셀 1에서 설정 완료
plt.rcParams['axes.unicode_minus'] = False

# 🚨 행(row) 개수를 그룹 수에 맞춰 4로 수정
groups = ['관광객', '직장인', '현지인', '대학생']
fig, axes = plt.subplots(len(groups), 2, figsize=(15, 20)) 

for i, g in enumerate(groups):
    # 해당 그룹 데이터 필터링
    data = bus_bts[bus_bts['user_type'] == g]
    
    # 1. 승차(Get-on) TOP 15
    top_geton = data['geton_station_name'].value_counts().head(15)
    axes[i, 0].barh(top_geton.index, top_geton.values, color='skyblue')
    axes[i, 0].set_title(f'[{g}] 승차 TOP 15', fontsize=14)
    axes[i, 0].invert_yaxis()  
    
    # 2. 하차(Get-off) TOP 15
    # 결측치(NaN) 제거 후 집계
    top_getoff = data['getoff_station_name'].dropna().value_counts().head(15)
    axes[i, 1].barh(top_getoff.index, top_getoff.values, color='salmon')
    axes[i, 1].set_title(f'[{g}] 하차 TOP 15', fontsize=14)
    axes[i, 1].invert_yaxis()  

plt.tight_layout()
plt.show()

### Chapter 5. 기상 이변이 대중교통 수요에 미치는 영향<br>
### (가설: 폭우 및 태풍 등 극단적 날씨는 버스 이용량의 예외적 변동(이상치)을 유발한다)

In [ ]:
# ============================================================
# [셀 11-1] 날씨(강수량/기온/풍속) vs 퇴근 승차 인원
# ============================================================

if 'date_str' not in train.columns:
    train['date_str'] = train['date_dt'].dt.strftime('%Y-%m-%d') if 'date_dt' in train.columns else train['date']

daily_ride = train.groupby('date_str')[TARGET_COL].sum().reset_index()
daily_ride.columns = ['date', TARGET_COL]

# 날씨 데이터 병합
daily_merged = pd.merge(daily_ride, weather_daily, on='date', how='left')
daily_merged = daily_merged.sort_values('date').reset_index(drop=True)

fig, axes = plt.subplots(3, 1, figsize=(18, 15))
fig.suptitle('날씨 vs 퇴근시간 승차 인원 분석', fontsize=16, fontweight='bold')

x = range(len(daily_merged))
xticks_labels = daily_merged['date'].str[5:]  # MM-DD 형식

# ① 강수량 vs 승차 인원
ax1 = axes[0]
ax2 = ax1.twinx()  # 오른쪽 y축 추가
bars = ax1.bar(x, daily_merged['강수량_합계'],
               color='skyblue', alpha=0.6, label='강수량(mm)', width=0.6)
ax2.plot(x, daily_merged[TARGET_COL],
         color='tomato', linewidth=2, marker='o', markersize=4, label='승차 인원')
ax1.set_xticks(list(x)[::2])
ax1.set_xticklabels(list(xticks_labels)[::2], rotation=45, fontsize=8)
ax1.set_ylabel('강수량 (mm)', color='skyblue')
ax2.set_ylabel('총 승차 인원', color='tomato')
ax1.set_title('일별 강수량 vs 퇴근 승차 인원')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

# ② 기온 vs 승차 인원
ax3 = axes[1]
ax4 = ax3.twinx()
ax3.plot(x, daily_merged['기온_평균'],
         color='orange', linewidth=2, marker='s', markersize=4, label='평균 기온(°C)')
ax4.plot(x, daily_merged[TARGET_COL],
         color='tomato', linewidth=2, marker='o', markersize=4, label='승차 인원')
ax3.set_xticks(list(x)[::2])
ax3.set_xticklabels(list(xticks_labels)[::2], rotation=45, fontsize=8)
ax3.set_ylabel('평균 기온 (°C)', color='orange')
ax4.set_ylabel('총 승차 인원', color='tomato')
ax3.set_title('일별 평균 기온 vs 퇴근 승차 인원')
lines3, labels3 = ax3.get_legend_handles_labels()
lines4, labels4 = ax4.get_legend_handles_labels()
ax3.legend(lines3 + lines4, labels3 + labels4, loc='upper left')

# ③ 풍속 vs 승차 인원
ax5 = axes[2]
ax6 = ax5.twinx()
ax5.plot(x, daily_merged['풍속_최대'],
         color='green', linewidth=2, marker='^', markersize=4, label='최대 풍속(m/s)')
ax6.plot(x, daily_merged[TARGET_COL],
         color='tomato', linewidth=2, marker='o', markersize=4, label='승차 인원')
ax5.set_xticks(list(x)[::2])
ax5.set_xticklabels(list(xticks_labels)[::2], rotation=45, fontsize=8)
ax5.set_ylabel('최대 풍속 (m/s)', color='green')
ax6.set_ylabel('총 승차 인원', color='tomato')
ax5.set_title('일별 최대 풍속 vs 퇴근 승차 인원')
lines5, labels5 = ax5.get_legend_handles_labels()
lines6, labels6 = ax6.get_legend_handles_labels()
ax5.legend(lines5 + lines6, labels5 + labels6, loc='upper left')

plt.tight_layout()
plt.savefig('09_weather_vs_ride.png', dpi=150, bbox_inches='tight')
plt.show()

=======================================================================================================

=======================================================================================================

---
# 3. 전처리

| 전처리 항목 | 내용 |
|------------|------|
| 기상 데이터 | 시간별 → 일별 집계 (기온/강수량/풍속/일조시간) |
| 저녁 기상 | 18~20시만 별도 집계 (예측 시간대 날씨) |
| 날짜 피처 | 요일, 주말, 공휴일(추석/개천절/한글날), 태풍(9/22~23) |
| in_out | '시내' → 0, '시외' → 1 인코딩 |
| BTS merge | bus_bts.csv에서 정류장/노선별 집계 후 merge, 결측 → 0 |
| 타겟 변환 | log1p 변환 (극단적 쏠림 완화, 예측 후 expm1으로 복원) |


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

#2. station_name을 집합(Set) 자료형으로 변환
train_stations = set(train['station_name'].dropna())
test_stations = set(test['station_name'].dropna())

# 3. 교집합 및 차집합 계산
only_in_train = train_stations - test_stations # Train에만 있는 역
only_in_test = test_stations - train_stations  # Test에만 있는 역
in_both = train_stations & test_stations       # 양쪽 다 있는 역

# 4. 개수 계산
counts = [len(only_in_train), len(in_both), len(only_in_test)]
labels = ['Only in Train', 'In Both', 'Only in Test']
colors = ['#ff9999', '#66b3ff', '#99ff99']

# 5. 시각화 (막대 그래프와 파이 차트)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# 막대 그래프
bars = ax1.bar(labels, counts, color=colors)
ax1.set_title('Station Name Distribution')
ax1.set_ylabel('Number of Unique Stations')

# 막대 그래프 위에 정확한 숫자 표시
for bar in bars:
    yval = bar.get_height()
    ax1.text(bar.get_x() + bar.get_width()/2, yval + 0.5, str(yval), ha='center', va='bottom')

# 파이 차트
ax2.pie(counts, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
ax2.set_title('Proportion of Station Names')

plt.suptitle('Train vs Test Station Name Comparison', fontsize=16)
plt.tight_layout()
plt.show()

# 💡 보너스: Test 데이터에만 있어서 문제가 되는 역의 이름 직접 확인하기
print(f"Test 데이터에만 존재하는 역 개수: {len(only_in_test)}개")
print("Test에만 있는 역 목록:", only_in_test)

---
# 4. 특성 엔지니어링 (팀원 5명 피처 통합)

팀원 5명이 각자 만든 피처를 분석한 결과, **핵심 아이디어의 80%가 겹쳤습니다.**
이는 팀 전체가 올바른 방향으로 접근했다는 증거입니다.

### 피처 구성
| 카테고리 | 피처 수 | 담당 팀원 | 핵심 피처 |
|----------|---------|----------|----------|
| 승차 원본+파생 | 14개 | 팀원 1,3,4,5 공통 | ride_total, ride_6_8 등 |
| 하차 파생 | 5개 | 팀원 4 | ride_takeoff_ratio 등 |
| 순유입 (net_flow) | 3개 | 팀원 2 | 승차-하차 = 방향성 |
| Target Encoding | 3개 | 팀원 1,3,5 공통 | te_route, te_station 등 |
| BTS 카드 | 10개 | 팀원 2 핵심 | bts_morning_getoff, bus_interval |
| 정류장 통계 | 4개 | 팀원 5 | station_cv 등 |
| 날짜 | 5개 | 팀원 4 | dayofweek, is_holiday 등 |
| 날씨 | 7개 | 공통 | rain_sum, temp_mean 등 |
| 상호작용 | 3개 | 공통 | commuter_x_weekend 등 |
| 기타 | 1개 | 공통 | in_out_code |
| **기존 합계** | **55개** | | |
| 관광객/현지인 비율 | 2개 | **팀원 4 고유** | usertype_tourist, usertype_local |
| 동 중심 거리 | 1개 | **팀원 4 고유** | dist_to_dong |
| 경과일수 | 1개 | **팀원 5 고유** | passed_days |
| **최종 합계** | **59개** | | |

### 가장 효과적이었던 피처 3가지
1. **bts_morning_getoff** (팀원 2): 아침에 여기서 내린 사람 = 저녁에 여기서 다시 타는 사람
2. **bus_interval** (팀원 2): 버스가 얼마나 자주 오는가 (공급 빈도 정보)
3. **net_flow** (팀원 2): 승차 - 하차 = 이 정류장이 출발지인가 도착지인가


In [ ]:
# ============================================================
# 셀 4. 기상 데이터 전처리
# ============================================================

def process_weather(df):
    df = df.copy()
    df.columns = ['지점', '지점명', '일시', '기온', '강수량', '풍속', '일조']
    df['일시'] = pd.to_datetime(df['일시'])
    df['date'] = df['일시'].dt.strftime('%Y-%m-%d')
    df['hour'] = df['일시'].dt.hour
    for col in ['기온', '강수량', '풍속', '일조']:
        df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0)
    return df

weather_sep = process_weather(weather_sep)
weather_oct = process_weather(weather_oct)
weather_all = pd.concat([weather_sep, weather_oct], ignore_index=True)
print(f'기상 데이터: {weather_all.shape}')

In [ ]:
# ============================================================
# 셀 5. 기상 데이터 일별 집계 (plan6 동일)
# ============================================================

weather_daily = weather_all.groupby('date').agg(
    temp_mean=('기온', 'mean'), temp_max=('기온', 'max'), temp_min=('기온', 'min'),
    rain_sum=('강수량', 'sum'), rain_max=('강수량', 'max'),
    wind_mean=('풍속', 'mean'), wind_max=('풍속', 'max'),
    sunshine=('일조', 'sum')
).reset_index()
weather_daily['is_rainy'] = (weather_daily['rain_sum'] > 0).astype(int)

weather_evening = weather_all[weather_all['hour'].between(18, 20)].groupby('date').agg(
    temp_evening=('기온', 'mean'), rain_evening=('강수량', 'sum'), wind_evening=('풍속', 'mean')
).reset_index()

weather_feat = weather_daily.merge(weather_evening, on='date', how='left').fillna(0)
print(f'기상 피처: {weather_feat.shape}')

In [ ]:
# ============================================================
# 셀 6. 날짜 피처 생성 (plan6 동일)
# ============================================================

def add_date_features(df):
    df = df.copy()
    df['date_dt'] = pd.to_datetime(df['date'])
    df['dayofweek'] = df['date_dt'].dt.dayofweek
    df['is_weekend'] = (df['dayofweek'] >= 5).astype(int)
    df['week_of_month'] = (df['date_dt'].dt.day - 1) // 7
    holidays = ['2019-09-12', '2019-09-13', '2019-09-14', '2019-10-03', '2019-10-09']
    df['is_holiday'] = df['date'].isin(holidays).astype(int)
    typhoon_dates = ['2019-09-06', '2019-09-07', '2019-09-22', '2019-09-23']
    df['is_typhoon'] = df['date'].isin(typhoon_dates).astype(int)
    return df

train = add_date_features(train)
test  = add_date_features(test)
print(f'Train: {train["date"].min()} ~ {train["date"].max()}')
print(f'Test:  {test["date"].min()} ~ {test["date"].max()}')

In [ ]:
# ============================================================
# 셀 7. 기상 데이터 merge (plan6 동일)
# ============================================================

train = train.merge(weather_feat, on='date', how='left')
test  = test.merge(weather_feat, on='date', how='left')
weather_cols = [c for c in weather_feat.columns if c != 'date']
train[weather_cols] = train[weather_cols].fillna(0)
test[weather_cols]  = test[weather_cols].fillna(0)
print('기상 merge 완료')

In [ ]:
# ============================================================
# 셀 8. [변경] 승차 + 하차 피처 (plan6 + 순유입 net_flow 추가)
# ============================================================
# [plan9 추가] net_flow = ride - takeoff (시간대별)
#   양수 = 이 정류장에서 타는 사람이 더 많음 (출발지, 주거지)
#   음수 = 이 정류장에서 내리는 사람이 더 많음 (도착지, 직장/학교)
#   → 음수인 곳에서 저녁에 승차가 많을 확률 높음
#   기존 ride_takeoff_ratio는 전체 비율만 → net_flow는 시간대별 방향성

ride_cols_1h = ['6~7_ride', '7~8_ride', '8~9_ride', '9~10_ride', '10~11_ride', '11~12_ride']
takeoff_cols_1h = ['6~7_takeoff', '7~8_takeoff', '8~9_takeoff', '9~10_takeoff', '10~11_takeoff', '11~12_takeoff']

def add_ride_features(df):
    df = df.copy()
    # === plan6 승차 피처 (동일) ===
    df['ride_total'] = df[ride_cols_1h].sum(axis=1)
    df['late_morning_ride'] = df['10~11_ride'] + df['11~12_ride']
    df['peak_ride'] = df['7~8_ride'] + df['8~9_ride']
    df['ride_max'] = df[ride_cols_1h].max(axis=1)
    df['ride_std'] = df[ride_cols_1h].std(axis=1)
    df['ride_6_8'] = df['6~7_ride'] + df['7~8_ride']
    df['ride_8_10'] = df['8~9_ride'] + df['9~10_ride']
    df['ride_10_12'] = df['10~11_ride'] + df['11~12_ride']

    # === plan6 하차 피처 (동일) ===
    df['takeoff_total'] = df[takeoff_cols_1h].sum(axis=1)
    df['peak_takeoff'] = df['7~8_takeoff'] + df['8~9_takeoff']
    df['late_takeoff'] = df['10~11_takeoff'] + df['11~12_takeoff']
    df['ride_takeoff_ratio'] = df['ride_total'] / (df['takeoff_total'] + 1)
    ride_mean = df[ride_cols_1h].mean(axis=1)
    df['morning_cv'] = df[ride_cols_1h].std(axis=1) / (ride_mean + 0.1)

    # === [plan9 신규] 시간대별 순유입 (ride - takeoff) ===
    df['net_flow_6_8'] = (df['6~7_ride'] + df['7~8_ride']) - (df['6~7_takeoff'] + df['7~8_takeoff'])
    df['net_flow_8_10'] = (df['8~9_ride'] + df['9~10_ride']) - (df['8~9_takeoff'] + df['9~10_takeoff'])
    df['net_flow_10_12'] = (df['10~11_ride'] + df['11~12_ride']) - (df['10~11_takeoff'] + df['11~12_takeoff'])

    return df

train = add_ride_features(train)
test  = add_ride_features(test)

net_flow_features = ['net_flow_6_8', 'net_flow_8_10', 'net_flow_10_12']

print('승차+하차 피처 생성 완료')
print(f'[plan9 신규] 순유입 피처:')
for f in net_flow_features:
    vals = train[f]
    print(f'  {f}: mean={vals.mean():.2f}, 음수비율={( vals < 0).mean():.1%}')

In [ ]:
# ============================================================
# 셀 9. in_out 인코딩 (plan6 동일)
# ============================================================

train['in_out_code'] = (train['in_out'] == '시외').astype(int)
test['in_out_code']  = (test['in_out'] == '시외').astype(int)
print('in_out 인코딩 완료')

In [ ]:
# ============================================================
# 셀 10. [변경] BTS 피처 (plan6 8개 + morning_getoff + bus_interval = 10개)
# ============================================================
# [plan9 추가]
#   bts_morning_getoff: BTS 오전(6~11시) 하차 인원 (route+station 레벨)
#     "아침에 이 노선의 이 정류장에서 내린 사람" = 저녁에 다시 탈 사람
#     train 데이터의 takeoff와 다른 소스 (BTS), route+station 조합이므로 TE와 다른 정보
#
#   bus_interval: BTS 타임스탬프 기반 노선별 평균 배차간격(분)
#     te_route = "이 노선의 수요가 얼마인지"
#     bus_interval = "이 노선이 얼마나 자주 오는지" → 다른 축의 정보

# --- plan6 BTS 피처 (동일) ---
bts_station = bus_bts.groupby('geton_station_code').agg(
    bts_total_users=('user_count', 'sum'),
    bts_trip_count=('user_count', 'count')
).reset_index()
bts_station.rename(columns={'geton_station_code': 'station_code'}, inplace=True)

for cat_id, cat_name in [(1, 'general'), (2, 'child'), (4, 'youth'), (6, 'senior')]:
    cat_count = bus_bts[bus_bts['user_category'] == cat_id].groupby(
        'geton_station_code')['user_count'].sum().reset_index()
    cat_count.columns = ['station_code', f'bts_{cat_name}_count']
    bts_station = bts_station.merge(cat_count, on='station_code', how='left')
bts_station = bts_station.fillna(0)
bts_station['bts_senior_ratio'] = bts_station['bts_senior_count'] / (bts_station['bts_total_users'] + 1)
bts_station['bts_youth_ratio'] = bts_station['bts_youth_count'] / (bts_station['bts_total_users'] + 1)

bts_getoff = bus_bts.groupby('getoff_station_code').agg(
    bts_getoff_total=('user_count', 'sum')).reset_index()
bts_getoff.rename(columns={'getoff_station_code': 'station_code'}, inplace=True)

bts_route_station = bus_bts.groupby(
    ['bus_route_id', 'geton_station_code']).agg(
    bts_rs_users=('user_count', 'sum')).reset_index()
bts_route_station.rename(columns={'geton_station_code': 'station_code'}, inplace=True)

# 통근자 비율 (plan6 동일)
card_days = bus_bts.groupby(['geton_station_code', 'user_card_id'])['geton_date'].nunique().reset_index()
card_days.columns = ['station_code', 'user_card_id', 'usage_days']
card_days['is_commuter'] = (card_days['usage_days'] >= 5).astype(int)
commuter_stats = card_days.groupby('station_code').agg(
    total_cards=('user_card_id', 'count'),
    commuter_cards=('is_commuter', 'sum')
).reset_index()
commuter_stats['bts_commuter_ratio'] = commuter_stats['commuter_cards'] / (commuter_stats['total_cards'] + 1)
commuter_stats = commuter_stats[['station_code', 'bts_commuter_ratio']]

# OD 밸런스 (plan6 동일)
bts_geton_total = bus_bts.groupby('geton_station_code')['user_count'].sum().reset_index()
bts_geton_total.columns = ['station_code', 'geton_total']
bts_getoff_total_od = bus_bts.groupby('getoff_station_code')['user_count'].sum().reset_index()
bts_getoff_total_od.columns = ['station_code', 'getoff_total_od']
od_balance = bts_geton_total.merge(bts_getoff_total_od, on='station_code', how='outer').fillna(0)
od_balance['bts_od_balance'] = od_balance['geton_total'] / (od_balance['geton_total'] + od_balance['getoff_total_od'] + 1)
od_balance = od_balance[['station_code', 'bts_od_balance']]

# 노선 다양성 (plan6 동일)
route_diversity = bus_bts.groupby('geton_station_code')['bus_route_id'].nunique().reset_index()
route_diversity.columns = ['station_code', 'bts_route_diversity']

# --- [plan9 신규] BTS 오전 하차 (route+station 레벨) ---
morning_getoff = bus_bts[bus_bts['getoff_time'].between('06:00:00', '11:00:00')]
getoff_agg = morning_getoff.groupby(
    ['bus_route_id', 'getoff_station_code'])['user_count'].sum().reset_index()
getoff_agg.columns = ['bus_route_id', 'station_code', 'bts_morning_getoff']
print(f'[plan9 신규] bts_morning_getoff: {getoff_agg.shape[0]}개 route+station 조합')
print(f'  mean={getoff_agg["bts_morning_getoff"].mean():.1f}, max={getoff_agg["bts_morning_getoff"].max()}')

# --- [plan9 신규] 배차간격 ---
bus_bts_temp = bus_bts.copy()
bus_bts_temp['geton_datetime'] = pd.to_datetime(
    bus_bts_temp['geton_date'] + ' ' + bus_bts_temp['geton_time']
)
bus_bts_temp = bus_bts_temp.sort_values('geton_datetime')
bus_bts_temp['time_diff'] = bus_bts_temp.groupby(
    ['geton_date', 'geton_station_code', 'bus_route_id']
)['geton_datetime'].diff()
bus_bts_temp['interval_min'] = bus_bts_temp['time_diff'].dt.total_seconds() / 60
# 합리적 범위만 (3분~180분)
valid_intervals = bus_bts_temp[(bus_bts_temp['interval_min'] > 3) & (bus_bts_temp['interval_min'] < 180)]
bus_interval_df = valid_intervals.groupby('bus_route_id')['interval_min'].mean().reset_index()
bus_interval_df.columns = ['bus_route_id', 'bus_interval']
bus_interval_df['bus_interval'] = bus_interval_df['bus_interval'].round(1)
print(f'[plan9 신규] bus_interval: {bus_interval_df.shape[0]}개 노선')
print(f'  mean={bus_interval_df["bus_interval"].mean():.1f}분, median={bus_interval_df["bus_interval"].median():.1f}분')
del bus_bts_temp, valid_intervals
gc.collect()

# --- Merge ---
bts_feat_station = bts_station[['station_code', 'bts_total_users', 'bts_senior_ratio', 'bts_youth_ratio']]
train = train.merge(bts_feat_station, on='station_code', how='left')
test  = test.merge(bts_feat_station, on='station_code', how='left')
train = train.merge(bts_getoff, on='station_code', how='left')
test  = test.merge(bts_getoff, on='station_code', how='left')
train = train.merge(bts_route_station, on=['bus_route_id', 'station_code'], how='left')
test  = test.merge(bts_route_station, on=['bus_route_id', 'station_code'], how='left')
train = train.merge(commuter_stats, on='station_code', how='left')
test  = test.merge(commuter_stats, on='station_code', how='left')
train = train.merge(od_balance, on='station_code', how='left')
test  = test.merge(od_balance, on='station_code', how='left')
train = train.merge(route_diversity, on='station_code', how='left')
test  = test.merge(route_diversity, on='station_code', how='left')
# [plan9 신규]
train = train.merge(getoff_agg, on=['bus_route_id', 'station_code'], how='left')
test  = test.merge(getoff_agg, on=['bus_route_id', 'station_code'], how='left')
train = train.merge(bus_interval_df, on='bus_route_id', how='left')
test  = test.merge(bus_interval_df, on='bus_route_id', how='left')

bts_features = ['bts_total_users', 'bts_senior_ratio', 'bts_youth_ratio',
                'bts_getoff_total', 'bts_rs_users',
                'bts_commuter_ratio', 'bts_od_balance', 'bts_route_diversity',
                'bts_morning_getoff', 'bus_interval']  # [변경] 8→10개
train[bts_features] = train[bts_features].fillna(0)
test[bts_features]  = test[bts_features].fillna(0)

del card_days, commuter_stats, od_balance, route_diversity, getoff_agg, bus_interval_df
gc.collect()
print(f'\nBTS merge 완료 (plan6: 8개 → plan9: {len(bts_features)}개)')

In [ ]:
# ============================================================
# 셀 11. 정류장 특성 피처 (plan6 동일)
# ============================================================

station_stats = train.groupby('station_code').agg(
    s_ride_mean=('ride_total', 'mean'),
    s_ride_std=('ride_total', 'std'),
    s_takeoff_mean=('takeoff_total', 'mean'),
).reset_index()
station_stats['station_morning_mean'] = station_stats['s_ride_mean']
station_stats['station_morning_cv'] = station_stats['s_ride_std'] / (station_stats['s_ride_mean'] + 0.1)
station_stats['station_takeoff_ratio'] = station_stats['s_takeoff_mean'] / (station_stats['s_ride_mean'] + station_stats['s_takeoff_mean'] + 0.1)

weekday_mean = train[train['is_weekend'] == 0].groupby('station_code')['ride_total'].mean()
weekend_mean = train[train['is_weekend'] == 1].groupby('station_code')['ride_total'].mean()
wd_ratio = weekday_mean / (weekday_mean + weekend_mean + 0.1)
wd_ratio = wd_ratio.reset_index()
wd_ratio.columns = ['station_code', 'station_weekday_ratio']

station_feat = station_stats[['station_code', 'station_morning_mean', 'station_morning_cv', 'station_takeoff_ratio']]
station_feat = station_feat.merge(wd_ratio, on='station_code', how='left')

train = train.merge(station_feat, on='station_code', how='left')
test  = test.merge(station_feat, on='station_code', how='left')

station_features = ['station_morning_mean', 'station_morning_cv', 'station_weekday_ratio', 'station_takeoff_ratio']
train[station_features] = train[station_features].fillna(0)
test[station_features]  = test[station_features].fillna(0)

print('정류장 특성 피처 생성 완료 (plan6 동일, 4개)')

In [ ]:
# ============================================================
# 셀 12. 상호작용 피처 (plan6 동일)
# ============================================================

def add_interaction_features(df):
    df = df.copy()
    df['commuter_x_weekend'] = df['bts_commuter_ratio'] * df['is_weekend']
    df['tourist_x_rain'] = (1 - df['bts_commuter_ratio']) * df['rain_sum']
    df['takeoff_x_weekend'] = df['station_takeoff_ratio'] * df['is_weekend']
    return df

train = add_interaction_features(train)
test  = add_interaction_features(test)

interaction_features = ['commuter_x_weekend', 'tourist_x_rain', 'takeoff_x_weekend']
print(f'상호작용 피처 {len(interaction_features)}개 생성 완료 (plan6 동일)')

In [ ]:
# ============================================================
# 셀 13. Target Encoding 함수 정의 (plan6 동일)
# ============================================================

N_FOLDS_TE = 5

def target_encode_smoothed_kfold(train_df, test_df, col, target,
                                  m=30, n_folds=N_FOLDS_TE, seed=SEED):
    global_mean = train_df[target].mean()
    train_encoded = pd.Series(np.nan, index=train_df.index)
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    for train_idx, val_idx in kf.split(train_df):
        fold_train = train_df.iloc[train_idx]
        group_stats = fold_train.groupby(col)[target].agg(['mean', 'count'])
        smoothed_map = (group_stats['count'] * group_stats['mean'] + m * global_mean) / (group_stats['count'] + m)
        train_encoded.iloc[val_idx] = train_df.iloc[val_idx][col].map(smoothed_map)
    train_encoded = train_encoded.fillna(global_mean)
    full_stats = train_df.groupby(col)[target].agg(['mean', 'count'])
    full_smoothed = (full_stats['count'] * full_stats['mean'] + m * global_mean) / (full_stats['count'] + m)
    test_encoded = test_df[col].map(full_smoothed).fillna(global_mean)
    return train_encoded, test_encoded

train['route_station'] = train['bus_route_id'].astype(str) + '_' + train['station_code'].astype(str)
test['route_station']  = test['bus_route_id'].astype(str) + '_' + test['station_code'].astype(str)
print('TE 함수 정의 완료')

In [ ]:
# ============================================================
# 셀 14. TE 적용 (plan6 동일, m=20)
# ============================================================

SMOOTHING_M = 20

train['te_route_station'], test['te_route_station'] = target_encode_smoothed_kfold(
    train, test, 'route_station', TARGET, m=SMOOTHING_M)
train['te_station'], test['te_station'] = target_encode_smoothed_kfold(
    train, test, 'station_code', TARGET, m=SMOOTHING_M)
train['te_route'], test['te_route'] = target_encode_smoothed_kfold(
    train, test, 'bus_route_id', TARGET, m=SMOOTHING_M)
print(f'TE 적용 완료 (m={SMOOTHING_M})')

In [ ]:
# ============================================================
# 셀 14-1. [TeamPlan1] 팀원 고유 피처 4개 생성
# ============================================================
# 5명의 피처를 분석한 결과, 기존 55개와 중복되지 않는 고유 피처 4개

# --- [팀원4] usertype_관광객 / user_type_현지인 ---
# BTS 카드의 user_category에서 관광객/현지인 비율 추출
# 관광 정류장 vs 통근 정류장을 구분하는 새로운 정보 축
#
# user_category 값:
#   1=일반, 2=어린이, 4=청소년, 6=경로
#   → 일반+경로 = 현지인 (정기 이용), 어린이+청소년 = 학생
#   → 관광객은 직접 구분이 어려우므로, 이용 빈도로 구분
#     카드 사용 1~2회 = 관광객 가능성 높음
#     카드 사용 5회+ = 현지인(통근자)

card_usage = bus_bts.groupby(['geton_station_code', 'user_card_id']).agg(
    usage_count=('user_count', 'count')
).reset_index()

# 1~2회 이용 = 관광객 추정, 5회+ = 현지인 추정
card_usage['is_tourist'] = (card_usage['usage_count'] <= 2).astype(int)
card_usage['is_local'] = (card_usage['usage_count'] >= 5).astype(int)

tourist_ratio = card_usage.groupby('geton_station_code').agg(
    total_cards=('user_card_id', 'count'),
    tourist_cards=('is_tourist', 'sum'),
    local_cards=('is_local', 'sum')
).reset_index()
tourist_ratio['usertype_tourist'] = tourist_ratio['tourist_cards'] / (tourist_ratio['total_cards'] + 1)
tourist_ratio['usertype_local'] = tourist_ratio['local_cards'] / (tourist_ratio['total_cards'] + 1)
tourist_ratio = tourist_ratio[['geton_station_code', 'usertype_tourist', 'usertype_local']]
tourist_ratio.rename(columns={'geton_station_code': 'station_code'}, inplace=True)

train = train.merge(tourist_ratio, on='station_code', how='left')
test = test.merge(tourist_ratio, on='station_code', how='left')
train[['usertype_tourist', 'usertype_local']] = train[['usertype_tourist', 'usertype_local']].fillna(0)
test[['usertype_tourist', 'usertype_local']] = test[['usertype_tourist', 'usertype_local']].fillna(0)

print(f'관광객 비율: mean={train["usertype_tourist"].mean():.4f}')
print(f'현지인 비율: mean={train["usertype_local"].mean():.4f}')

# --- [팀원4] dist_to_dong (동 중심지까지 거리) ---
# 제주시청 좌표를 동 중심지 대표점으로 사용
# 도심에 가까울수록 퇴근 수요 높음
import math

def haversine(lat1, lon1, lat2, lon2):
    R = 6371
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    return R * 2 * math.asin(math.sqrt(a))

# 제주시 중심 (제주시청)
jeju_center = (33.4996, 126.5312)

for df in [train, test]:
    df['dist_to_dong'] = df.apply(
        lambda row: haversine(row['latitude'], row['longitude'], jeju_center[0], jeju_center[1]), axis=1)

print(f'동 중심 거리: mean={train["dist_to_dong"].mean():.2f}km, max={train["dist_to_dong"].max():.2f}km')

# --- [팀원5] passed_days (경과일수) ---
# 9/1부터 며칠째인지. 시간 흐름에 따른 변화를 모델이 학습 가능
# train: 0~29 (9월), test: 30~45 (10월)
base_date = pd.Timestamp('2019-09-01')
train['passed_days'] = (train['date_dt'] - base_date).dt.days
test['passed_days'] = (test['date_dt'] - base_date).dt.days

print(f'경과일수: train={train["passed_days"].min()}~{train["passed_days"].max()}, '
      f'test={test["passed_days"].min()}~{test["passed_days"].max()}')

team_features = ['usertype_tourist', 'usertype_local', 'dist_to_dong', 'passed_days']
print(f'\n팀원 고유 피처 {len(team_features)}개 생성 완료')

del card_usage, tourist_ratio
gc.collect()


In [ ]:
# ============================================================
# 셀 15. [변경] 최종 피처 선택 — 55개 & 59개 모두 준비
# ============================================================

ride_features = ['6~7_ride', '7~8_ride', '8~9_ride', '9~10_ride', '10~11_ride', '11~12_ride',
                 'ride_6_8', 'ride_8_10', 'ride_10_12']
ride_derived = ['ride_total', 'late_morning_ride', 'peak_ride', 'ride_max', 'ride_std']
takeoff_features = ['takeoff_total', 'peak_takeoff', 'late_takeoff', 'ride_takeoff_ratio', 'morning_cv']
te_features = ['te_route_station', 'te_station', 'te_route']
date_features = ['dayofweek', 'is_weekend', 'is_holiday', 'is_typhoon', 'week_of_month']
weather_features = ['rain_sum', 'rain_evening', 'wind_evening', 'is_rainy', 'sunshine', 'wind_max', 'temp_mean']
other_features = ['in_out_code']

# 55개 (기존 최고)
FEATURES_55 = (ride_features + ride_derived + takeoff_features +
               net_flow_features + te_features + date_features +
               weather_features + other_features +
               bts_features + station_features + interaction_features)

# 59개 (55 + 팀원 고유 4개)
FEATURES_59 = FEATURES_55 + team_features

# 55개 세팅
X_train_55 = train[FEATURES_55].copy()
X_test_55 = test[FEATURES_55].copy()

# 59개 세팅
X_train_59 = train[FEATURES_59].copy()
X_test_59 = test[FEATURES_59].copy()

# 공통
y_train = train[TARGET].copy()
y_train_log = np.log1p(y_train)

print(f'55개: X_train={X_train_55.shape}, X_test={X_test_55.shape}')
print(f'59개: X_train={X_train_59.shape}, X_test={X_test_59.shape}')
print(f'\n추가된 팀원 피처: {team_features}')


---
# 5. 모델링

### 모델링 발전 과정 (팀원 중 한 명의 개인 실험에서 시작)

| 단계 | 피처 | 모델 | private 결과 | 핵심 발견 |
|------|------|------|-------------|----------|
| plan9 | 55개 | LGB only | 단독 최고 | 55개가 최적 피처 수 |
| plan10~16 | 57~95개 | LGB only | 전부 하락 | 피처 추가 = 과적합 |
| plan17 | 95개 | LGB + **RF** | 개선! | **모델 다양성이 핵심** |
| plan18 | 55개 | LGB + RF | 더 개선! | 55개 + 모델 다양성 = 최적 |
| plan20 | 55+95 블렌딩 | LGB + RF | 최고점 | 다른 피처셋 블렌딩 효과 |
| plan22 | 55개 | LGB + RF + **XGB** + 시간CV | 최고점 갱신 | 3모델 + 시간기반 튜닝 |

### 핵심 교훈
1. **피처 수 늘리기 ≠ 성능 향상** (작은 데이터에서는 과적합)
2. **모델 다양성이 가장 효과적** (LGB+RF+XGB 앙상블)
3. **시간 기반 CV**로 10월 일반화에 맞춘 하이퍼파라미터 튜닝
4. **서로 다른 피처셋의 블렌딩**이 추가 개선을 줌

### 현재 모델 구성
```
3종 모델 × 3 시드 × 5 폴드 = 45개 모델

LightGBM  (Optuna 시간기반 튜닝, GPU)
RandomForest (고정 파라미터, CPU)
XGBoost (고정 파라미터, CPU)

→ 55개 피처 모델 (45개) + 59개 피처 모델 (45개) = 총 90개 모델
→ 55개 vs 59개 비교 후 블렌딩
```


In [ ]:
# ============================================================
# 셀 16. Adversarial Validation (간소화)
# ============================================================
from sklearn.metrics import roc_auc_score

for name, X_tr, X_te in [('55개', X_train_55, X_test_55), ('59개', X_train_59, X_test_59)]:
    X_adv = pd.concat([X_tr, X_te], ignore_index=True)
    y_adv = np.array([0]*len(X_tr) + [1]*len(X_te))
    adv_model = lgb.LGBMClassifier(n_estimators=100, num_leaves=31, verbose=-1, random_state=SEED)
    kf = KFold(n_splits=3, shuffle=True, random_state=SEED)
    adv_oof = np.zeros(len(X_adv))
    for tr_idx, val_idx in kf.split(X_adv):
        adv_model.fit(X_adv.iloc[tr_idx], y_adv[tr_idx])
        adv_oof[val_idx] = adv_model.predict_proba(X_adv.iloc[val_idx])[:,1]
    auc = roc_auc_score(y_adv, adv_oof)
    print(f'{name} AV AUC: {auc:.4f}')
    del X_adv, y_adv, adv_oof
gc.collect()


In [ ]:
# ============================================================
# 셀 17. CV 프레임워크 + 시간 기반 split
# ============================================================

N_FOLDS = 5

def evaluate_cv(model_fn, X, y_log, y_orig, n_folds=N_FOLDS, seed=SEED):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=seed)
    oof_preds = np.zeros(len(X))
    scores = []
    models = []
    for fold, (train_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_tr, y_val = y_log.iloc[train_idx], y_log.iloc[val_idx]
        model = model_fn(X_tr, y_tr, X_val, y_val)
        models.append(model)
        pred = np.clip(np.expm1(model.predict(X_val)), 0, None)
        oof_preds[val_idx] = pred
        fold_mae = mean_absolute_error(y_orig.iloc[val_idx], pred)
        scores.append(fold_mae)
    overall_mae = mean_absolute_error(y_orig, oof_preds)
    print(f'  CV MAE: {overall_mae:.6f} (+/-{np.std(scores):.6f})')
    return models, oof_preds, overall_mae

# 시간 기반 split (Optuna용)
time_train_mask = train['date_dt'] <= '2019-09-22'
time_val_mask = train['date_dt'] > '2019-09-22'
time_train_idx = np.where(time_train_mask)[0]
time_val_idx = np.where(time_val_mask)[0]
print(f'시간 split: train={len(time_train_idx):,}, val={len(time_val_idx):,}')


In [ ]:
# ============================================================
# Step 1. 55개 피처 모델 (베이스라인) — LGB GPU / XGB CPU
# ============================================================
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb

print('=' * 70)
print('  Step 1: 55개 피처 (기존 최고 성능 베이스라인)')
print('=' * 70)

SEEDS = [42, 123, 456]
X_train_use = X_train_55
X_test_use = X_test_55

# --- Optuna (시간 기반, GPU) ---
def objective_lgb_55(trial):
    params = {
        'objective': 'regression_l1', 'metric': 'mae', 'boosting_type': 'gbdt',
        'device': 'gpu',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 95),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'n_estimators': 3000, 'random_state': SEED, 'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train_use.iloc[time_train_idx], y_train_log.iloc[time_train_idx],
              eval_set=[(X_train_use.iloc[time_val_idx], y_train_log.iloc[time_val_idx])],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    pred = np.clip(np.expm1(model.predict(X_train_use.iloc[time_val_idx])), 0, None)
    return mean_absolute_error(y_train.iloc[time_val_idx], pred)

study_55 = optuna.create_study(direction='minimize', study_name='lgb_55')
study_55.optimize(objective_lgb_55, n_trials=15, show_progress_bar=True)
best_params_55 = study_55.best_params
print(f'LGB-55 Best MAE: {study_55.best_value:.6f}')

# --- LGB 3-seed (GPU) ---
print('\n--- LGB 55개 ---')
lgb55_models, lgb55_oofs = [], []
for seed in SEEDS:
    def train_lgb(X_tr, y_tr, X_val, y_val, _s=seed):
        m = lgb.LGBMRegressor(objective='regression_l1', metric='mae', device='gpu',
                              n_estimators=3000, random_state=_s, verbose=-1, **best_params_55)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        return m
    models, oof, _ = evaluate_cv(train_lgb, X_train_use, y_train_log, y_train, seed=seed)
    lgb55_models.append(models)
    lgb55_oofs.append(oof)

# --- RF 3-seed (CPU) ---
print('\n--- RF 55개 ---')
rf55_models, rf55_oofs = [], []
for seed in SEEDS:
    def train_rf(X_tr, y_tr, X_val, y_val, _s=seed):
        m = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_split=10,
                                  min_samples_leaf=5, max_features=0.7, n_jobs=-1, random_state=_s)
        m.fit(X_tr, y_tr)
        return m
    models, oof, _ = evaluate_cv(train_rf, X_train_use, y_train_log, y_train, seed=seed)
    rf55_models.append(models)
    rf55_oofs.append(oof)

# --- XGB 3-seed (CPU) ---
print('\n--- XGB 55개 ---')
xgb55_models, xgb55_oofs = [], []
for seed in SEEDS:
    def train_xgb(X_tr, y_tr, X_val, y_val, _s=seed):
        m = xgb.XGBRegressor(objective='reg:absoluteerror', n_estimators=3000, learning_rate=0.05,
                              max_depth=8, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                              tree_method='hist', random_state=_s, verbosity=0)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        return m
    models, oof, _ = evaluate_cv(train_xgb, X_train_use, y_train_log, y_train, seed=seed)
    xgb55_models.append(models)
    xgb55_oofs.append(oof)

# 55개 3모델 평균
all55_oofs = lgb55_oofs + rf55_oofs + xgb55_oofs
oof_55_avg = np.mean(all55_oofs, axis=0)
mae_55 = mean_absolute_error(y_train, oof_55_avg)
print(f'\n=== Step 1 결과: 55개 3모델 평균 MAE = {mae_55:.6f} ===')


## [참고] 기존 XGB GPU 오류 관련
기존에는 XGBoost GPU에서 오류가 발생하여 별도 셀로 분리 실행했으나,
현재 버전에서는 CPU(`tree_method='hist'`)로 수정되어 정상 실행됩니다.


In [ ]:
# ============================================================
# [참고] 기존에 XGB GPU 에러로 분리 실행했던 셀
# Step 1 셀에서 이미 CPU(tree_method='hist')로 수정되어 불필요
# 이 셀은 건너뛰어도 됩니다
# ============================================================
print('Step 1에서 이미 XGB CPU로 실행 완료. 이 셀은 건너뛰어도 됩니다.')


In [ ]:
# ============================================================
# Feature Importance 시각화 (55개 피처 기준)
# ============================================================

# LGB 모델 중 첫 번째 시드, 첫 번째 폴드의 중요도 사용
fi_model = lgb55_models[0][0]
fi = pd.DataFrame({
    'feature': FEATURES_55,
    'importance': fi_model.feature_importances_
}).sort_values('importance', ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(20, 10))

# 상위 20개
top20 = fi.head(20)
axes[0].barh(range(len(top20)), top20['importance'].values, color='steelblue')
axes[0].set_yticks(range(len(top20)))
axes[0].set_yticklabels(top20['feature'].values)
axes[0].invert_yaxis()
axes[0].set_xlabel('Importance')
axes[0].set_title('Feature Importance Top 20 (LGB 55개)')

# 카테고리별 합산
category_map = {}
for f in FEATURES_55:
    if f in ['6~7_ride','7~8_ride','8~9_ride','9~10_ride','10~11_ride','11~12_ride',
             'ride_6_8','ride_8_10','ride_10_12','ride_total','late_morning_ride',
             'peak_ride','ride_max','ride_std']:
        category_map[f] = '승차 피처'
    elif f in ['takeoff_total','peak_takeoff','late_takeoff','ride_takeoff_ratio','morning_cv']:
        category_map[f] = '하차 피처'
    elif f.startswith('net_flow'):
        category_map[f] = '순유입'
    elif f.startswith('te_'):
        category_map[f] = 'Target Encoding'
    elif f.startswith('bts_') or f == 'bus_interval':
        category_map[f] = 'BTS 카드'
    elif f.startswith('station_'):
        category_map[f] = '정류장 통계'
    elif f in ['dayofweek','is_weekend','is_holiday','is_typhoon','week_of_month']:
        category_map[f] = '날짜'
    elif f in ['rain_sum','rain_evening','wind_evening','is_rainy','sunshine','wind_max','temp_mean']:
        category_map[f] = '날씨'
    elif f.endswith('_x_weekend') or f == 'tourist_x_rain':
        category_map[f] = '상호작용'
    else:
        category_map[f] = '기타'

fi['category'] = fi['feature'].map(category_map)
cat_fi = fi.groupby('category')['importance'].sum().sort_values(ascending=False)

colors = ['#e74c3c','#3498db','#2ecc71','#f39c12','#9b59b6','#1abc9c','#e67e22','#34495e','#95a5a6','#d35400']
axes[1].barh(range(len(cat_fi)), cat_fi.values, color=colors[:len(cat_fi)])
axes[1].set_yticks(range(len(cat_fi)))
axes[1].set_yticklabels(cat_fi.index)
axes[1].invert_yaxis()
axes[1].set_xlabel('Total Importance')
axes[1].set_title('카테고리별 Feature Importance 합산')

plt.tight_layout()
plt.show()

print('
=== Top 10 피처 ===')
for i, row in fi.head(10).iterrows():
    print(f"  {row['feature']:30s} {row['importance']:>8,d}  ({row['category']})")


In [ ]:
# ============================================================
# Step 2. 59개 피처 모델 (팀원 피처 추가) — LGB GPU / XGB CPU
# ============================================================

print('=' * 70)
print('  Step 2: 59개 피처 (55 + 팀원 고유 4개)')
print('=' * 70)

X_train_use = X_train_59
X_test_use = X_test_59

# --- Optuna (시간 기반, GPU) ---
def objective_lgb_59(trial):
    params = {
        'objective': 'regression_l1', 'metric': 'mae', 'boosting_type': 'gbdt',
        'device': 'gpu',
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 95),
        'max_depth': trial.suggest_int('max_depth', 4, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 20, 150),
        'subsample': trial.suggest_float('subsample', 0.6, 0.95),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.9),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.01, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.01, 10.0, log=True),
        'n_estimators': 3000, 'random_state': SEED, 'verbose': -1,
    }
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train_use.iloc[time_train_idx], y_train_log.iloc[time_train_idx],
              eval_set=[(X_train_use.iloc[time_val_idx], y_train_log.iloc[time_val_idx])],
              callbacks=[lgb.early_stopping(50), lgb.log_evaluation(0)])
    pred = np.clip(np.expm1(model.predict(X_train_use.iloc[time_val_idx])), 0, None)
    return mean_absolute_error(y_train.iloc[time_val_idx], pred)

study_59 = optuna.create_study(direction='minimize', study_name='lgb_59')
study_59.optimize(objective_lgb_59, n_trials=15, show_progress_bar=True)
best_params_59 = study_59.best_params
print(f'LGB-59 Best MAE: {study_59.best_value:.6f}')

# --- LGB 3-seed (GPU) ---
print('\n--- LGB 59개 ---')
lgb59_models, lgb59_oofs = [], []
for seed in SEEDS:
    def train_lgb(X_tr, y_tr, X_val, y_val, _s=seed):
        m = lgb.LGBMRegressor(objective='regression_l1', metric='mae', device='gpu',
                              n_estimators=3000, random_state=_s, verbose=-1, **best_params_59)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
        return m
    models, oof, _ = evaluate_cv(train_lgb, X_train_use, y_train_log, y_train, seed=seed)
    lgb59_models.append(models)
    lgb59_oofs.append(oof)

# --- RF 3-seed (CPU) ---
print('\n--- RF 59개 ---')
rf59_models, rf59_oofs = [], []
for seed in SEEDS:
    def train_rf(X_tr, y_tr, X_val, y_val, _s=seed):
        m = RandomForestRegressor(n_estimators=300, max_depth=15, min_samples_split=10,
                                  min_samples_leaf=5, max_features=0.7, n_jobs=-1, random_state=_s)
        m.fit(X_tr, y_tr)
        return m
    models, oof, _ = evaluate_cv(train_rf, X_train_use, y_train_log, y_train, seed=seed)
    rf59_models.append(models)
    rf59_oofs.append(oof)

# --- XGB 3-seed (CPU) ---
print('\n--- XGB 59개 ---')
xgb59_models, xgb59_oofs = [], []
for seed in SEEDS:
    def train_xgb(X_tr, y_tr, X_val, y_val, _s=seed):
        m = xgb.XGBRegressor(objective='reg:absoluteerror', n_estimators=3000, learning_rate=0.05,
                              max_depth=8, min_child_weight=20, subsample=0.8, colsample_bytree=0.8,
                              tree_method='hist', random_state=_s, verbosity=0)
        m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        return m
    models, oof, _ = evaluate_cv(train_xgb, X_train_use, y_train_log, y_train, seed=seed)
    xgb59_models.append(models)
    xgb59_oofs.append(oof)

# 59개 3모델 평균
all59_oofs = lgb59_oofs + rf59_oofs + xgb59_oofs
oof_59_avg = np.mean(all59_oofs, axis=0)
mae_59 = mean_absolute_error(y_train, oof_59_avg)
print(f'\n=== Step 2 결과: 59개 3모델 평균 MAE = {mae_59:.6f} ===')
print(f'55개 vs 59개: {mae_55:.6f} vs {mae_59:.6f} (차이: {mae_55 - mae_59:+.6f})')


In [ ]:
# ============================================================
# Step 3. 55개 vs 59개 블렌딩 + Test 예측 + 제출파일
# ============================================================
import os

print('=' * 70)
print('  Step 3: 블렌딩 + 제출파일 생성')
print('=' * 70)

# --- OOF 블렌딩 ---
print('\n=== OOF 블렌딩 비교 ===')
print(f'  55개 단독: {mae_55:.6f}')
print(f'  59개 단독: {mae_59:.6f}')

corr_55_59 = np.corrcoef(oof_55_avg, oof_59_avg)[0,1]
print(f'  55 vs 59 상관: {corr_55_59:.6f}')

for w55 in [0.5, 0.6, 0.7, 0.8]:
    w59 = 1 - w55
    blended_oof = w55 * oof_55_avg + w59 * oof_59_avg
    blend_mae = mean_absolute_error(y_train, blended_oof)
    print(f'  55:{w55} + 59:{w59:.1f} → {blend_mae:.6f}')

# --- Test 예측 ---
def predict_models(model_list, X):
    preds = np.zeros(len(X))
    n = 0
    for seed_models in model_list:
        for model in seed_models:
            preds += np.clip(np.expm1(model.predict(X)), 0, None)
            n += 1
    return preds / n

# 55개 test 예측
t55_lgb = predict_models(lgb55_models, X_test_55)
t55_rf = predict_models(rf55_models, X_test_55)
t55_xgb = predict_models(xgb55_models, X_test_55)
t55_avg = (t55_lgb + t55_rf + t55_xgb) / 3

# 59개 test 예측
t59_lgb = predict_models(lgb59_models, X_test_59)
t59_rf = predict_models(rf59_models, X_test_59)
t59_xgb = predict_models(xgb59_models, X_test_59)
t59_avg = (t59_lgb + t59_rf + t59_xgb) / 3

# --- 제출파일 ---
print('\n=== 제출파일 생성 ===')

# 55개 단독
s = submission[['id']].copy()
s['18~20_ride'] = np.round(t55_avg).astype(int).clip(0)
s.to_csv('submission_team1_55.csv', index=False)
print('submission_team1_55.csv (55개 3모델)')

# 59개 단독
s = submission[['id']].copy()
s['18~20_ride'] = np.round(t59_avg).astype(int).clip(0)
s.to_csv('submission_team1_59.csv', index=False)
print('submission_team1_59.csv (59개 3모델)')

# 55+59 블렌딩
for w55 in [0.5, 0.6, 0.7]:
    w59 = 1 - w55
    blended = w55 * t55_avg + w59 * t59_avg
    s = submission[['id']].copy()
    s['18~20_ride'] = np.round(blended).astype(int).clip(0)
    fname = f'submission_team1_blend_{int(w55*10)}_{int(w59*10)}.csv'
    s.to_csv(fname, index=False)
    print(f'{fname} (55:{w55} + 59:{w59:.1f})')

# v17과 3-way (기존 최고점 방식 확장)
if os.path.exists('submission_lgb_v17_avg.csv'):
    v17 = pd.read_csv('submission_lgb_v17_avg.csv')['18~20_ride'].values.astype(float)
    # 55:59:v17 = 5:2:3
    blend3 = 0.5 * np.round(t55_avg) + 0.2 * np.round(t59_avg) + 0.3 * v17
    s = submission[['id']].copy()
    s['18~20_ride'] = np.round(blend3).astype(int).clip(0)
    s.to_csv('submission_team1_3way.csv', index=False)
    print('submission_team1_3way.csv (55:0.5 + 59:0.2 + v17:0.3)')

print('\n제출파일 생성 완료!')


In [ ]:
# ============================================================
# 최종 요약
# ============================================================

print('=' * 70)
print('  TeamPlan1: 팀 피처 통합 검증 결과')
print('=' * 70)

print(f'\n=== 추가된 팀원 고유 피처 ===')
print(f'  usertype_tourist (팀원4): 관광객 비율')
print(f'  usertype_local (팀원4):   현지인 비율')
print(f'  dist_to_dong (팀원4):     동 중심 거리')
print(f'  passed_days (팀원5):      경과일수')

print(f'\n=== CV MAE 비교 ===')
print(f'  55개 (기존):     {mae_55:.6f}')
print(f'  59개 (55+팀원):  {mae_59:.6f}')
diff = mae_55 - mae_59
if diff > 0:
    print(f'  → 59개가 {diff:.6f} 개선!')
else:
    print(f'  → 55개가 {-diff:.6f} 더 좋음')

print(f'\n=== 55 vs 59 상관관계: {corr_55_59:.6f} ===')
if corr_55_59 < 0.99:
    print(f'  상관이 낮으므로 블렌딩 효과 기대됨')
else:
    print(f'  상관이 높아서 블렌딩 효과 제한적')

print(f'\n=== 제출파일 ===')
print(f'  submission_team1_55.csv         (55개 단독)')
print(f'  submission_team1_59.csv         (59개 단독)')
print(f'  submission_team1_blend_*.csv    (55+59 블렌딩)')
print(f'  submission_team1_3way.csv       (55+59+v17 3-way)')
print('=' * 70)


---
# 6. 최종 결과 & 제출


## 생성된 파일 제출결과, 제일 예측점수가 높았던 파일은
## submission_team1_3way.csv 였습니다.
## private 13등

---

## 최종 제출 결과 분석

### 가장 높은 점수: `submission_team1_3way.csv`

이 파일은 **3가지 서로 다른 모델의 예측을 섞은(3-way 블렌딩)** 결과입니다.

```
submission_team1_3way.csv = 
    55개 피처 모델 × 0.5    (기존 검증된 최고 피처셋)
  + 59개 피처 모델 × 0.2    (팀원 고유 피처 4개 추가)
  + 95개 피처 모델 × 0.3    (대량 피처 + LGB+RF)
```

### 왜 이 파일이 가장 높았는가

**1. 세 모델이 각각 다른 관점으로 예측했기 때문**

| 모델 | 피처 수 | 특징 | 역할 |
|------|---------|------|------|
| 55개 모델 | 55개 | 핵심 피처만 사용, 과적합 적음 | **안정적 베이스** (비중 50%) |
| 59개 모델 | 59개 | 55개 + 관광객/현지인/거리/경과일수 | **이용자 유형 정보 보완** (비중 20%) |
| 95개 모델 | 95개 | 요일별 TE + 시간대 집계 등 대량 피처 | **정교한 패턴 포착** (비중 30%) |

- 55개 모델이 "전체적으로 안정적인 예측"을 담당
- 59개 모델이 "관광 정류장 vs 통근 정류장 구분" 정보를 보완
- 95개 모델이 "특정 요일/시간대의 세밀한 패턴"을 포착

**2. 단일 모델의 한계를 블렌딩이 극복**

단일 모델은 자신의 피처셋에서만 학습하므로 **특정 영역에서 약점**이 생깁니다.
하지만 서로 다른 피처셋의 모델을 섞으면, 한 모델이 틀리는 부분을 다른 모델이 보완합니다.

```
55개 모델이 관광지 정류장 예측 실패 → 59개 모델(관광객 비율 피처)이 보완
95개 모델이 10월 일반화 실패 → 55개 모델(안정적)이 상쇄
```

**3. 비율이 핵심: 55개에 가장 높은 비중(50%)**

55개 모델이 private에서 가장 안정적이었기 때문에 50%로 가장 높은 비중을 두고,
나머지 모델은 **보완 역할**로 소량(20~30%) 배합한 것이 최적이었습니다.

### 핵심 교훈

> **최고 점수는 하나의 강력한 모델이 아니라, 서로 다른 관점을 가진 여러 모델의 조합에서 나왔습니다.**
> 
> 팀원 5명이 각자 다른 피처를 만든 것이 결과적으로 "다양한 관점"을 제공했고,
> 이를 적절한 비율로 블렌딩한 것이 최종 성능 향상의 핵심이었습니다.